# Session 2 · NLP Applied to Emotion Detection
## Activity 1 — From Raw Text to Tokens · **SOLUTION**

| Step | Topic |
|------|-------|
| **1.1** | Text as strings — lowercase + remove punctuation |
| **1.2** | Naive tokenisation — `.split()` and its limits |
| **1.3** | Character-by-character — full control, emoticon-aware |
| **1.4** | Complete pipeline — integrate everything |

---
## Activity 1.1 — Text as Strings

In [ ]:
sentence = "The Battery life on this Phone is AMAZING... it lasts 2 full days! :)"
print("Original:", sentence)

In [ ]:
# TODO 1 — lowercase
lower_sentence = sentence.lower()
print("Lowercase:", lower_sentence)

In [ ]:
# TODO 2 — remove punctuation
clean_chars = []
for char in lower_sentence:
    if char.isalpha() or char.isdigit() or char == " ":
        clean_chars.append(char)
clean_sentence = "".join(clean_chars)

print("BEFORE:", sentence)
print("AFTER: ", clean_sentence)

In [ ]:
# TODO 3 — clean_text function
def clean_text(text):
    result = []
    for char in text.lower():
        if char.isalpha() or char.isdigit() or char == " ":
            result.append(char)
    return "".join(result)

test_sentences = [
    "I LOVE this! Best product ever.",
    "Terrible... absolutely terrible. 0/10.",
    "Not bad :) would buy again!",
]
for s in test_sentences:
    print(f"  IN : {s}")
    print(f"  OUT: {clean_text(s)}")
    print()

**Answers to discussion questions:**

1. `:)` and `...` are lost entirely — both carry emotional information.
2. `!` signals emphasis/excitement; removing it loses intensity. `2` is ambiguous — matters in "5-star" vs "1-star".
3. `"1-star battery"` vs `"5-star battery"` — the digit completely changes sentiment.
4. Extra spaces are handled in Activity 1.2 using `.split()` without arguments.

---
## Activity 1.2 — Tokenisation: The Naive Approach

In [ ]:
# TODO 4 — tokenise before cleaning
raw_sentence = "I love NLP... and Python! It's amazing :)"
raw_tokens = raw_sentence.split(" ")
print("Raw tokens:", raw_tokens)

In [ ]:
# TODO 5 — tokenise after cleaning
clean_tokens = clean_text(raw_sentence).split(" ")
print("Clean tokens:", clean_tokens)

In [ ]:
# TODO 6 — compare split variants
tokens_single_space = clean_text(raw_sentence).split(" ")
tokens_any_space    = clean_text(raw_sentence).split()
print('split(" ") ->', tokens_single_space)
print("split()    ->", tokens_any_space)
# split() is better — removes empty strings from double spaces

In [ ]:
test_sentences_12 = [
    "Best. Product. Ever.",
    "I can't believe how bad this is!!!",
    "5-star experience, would recommend :)",
    "not bad, not great... just meh",
]
for s in test_sentences_12:
    print(f"Input : {s}")
    print(f"Tokens: {clean_text(s).split()}")
    print()

# ANSWER: ':)' in sentence 3 is lost. 'not' in sentence 4 is kept
# (which is good) but the emoticon loss is significant for emotion detection.

---
## Activity 1.3 — Tokenisation: Character by Character

In [ ]:
# TODO 7 — basic char-by-char tokeniser
def tokenise_basic(text):
    tokens = []
    buffer = ""
    for char in text.lower():
        if char == " ":
            if buffer:
                tokens.append(buffer)
                buffer = ""
        elif char.isalpha() or char.isdigit():
            buffer += char
    if buffer:
        tokens.append(buffer)
    return tokens

print(tokenise_basic("I love NLP... and Python! It's amazing :)"))

In [ ]:
# TODO 8 — emoticon-aware tokeniser
EMOTICONS = {":)", ":(", ":D", ":P", ";)", ":/", ":o", "XD", "<3"}

def tokenise_with_emoticons(text):
    tokens = []
    buffer = ""
    i = 0
    text = text.lower()
    while i < len(text):
        two_chars = text[i:i+2]
        if two_chars in EMOTICONS:
            if buffer.strip():
                tokens.append(buffer.strip())
                buffer = ""
            tokens.append(two_chars)
            i += 2
            continue
        char = text[i]
        if char == " ":
            if buffer.strip():
                tokens.append(buffer.strip())
            buffer = ""
        elif char.isalpha() or char.isdigit():
            buffer += char
        i += 1
    if buffer.strip():
        tokens.append(buffer.strip())
    return tokens

for s in ["I love NLP... and Python! It's amazing :)",
          "Best. Product. Ever. :D",
          "Terrible experience :( would not recommend",
          "Not bad ;) would buy again!"]:
    print(f"Input  : {s}")
    print(f"Basic  : {tokenise_basic(s)}")
    print(f"Emotion: {tokenise_with_emoticons(s)}")
    print()

In [ ]:
# TODO 9 — configurable tokeniser
def tokenise(text, keep_emoticons=True):
    if keep_emoticons:
        return tokenise_with_emoticons(text)
    return tokenise_basic(text)

s = "Great experience :) would totally recommend!"
print(f"keep_emoticons=True : {tokenise(s, True)}")
print(f"keep_emoticons=False: {tokenise(s, False)}")

**Answers:**

1. The model can learn `':)'` → positive sentiment. Split into `:` + `)` those characters appear in many contexts and carry no signal.
2. `!!!`, `...`, negations (`n't`), all-caps words (`AMAZING`) are good candidates.
3. `"don't"` → `["do", "not"]` is semantically clearest for sentiment; `["don't"]` preserves form; `["don", "t"]` loses both.

---
## Activity 1.4 — Complete Normalisation Pipeline

In [ ]:
EMOTICONS_14 = {":)", ":(", ":D", ":P", ";)", ":/", ":o", "XD", "<3",
                ":-)", ":-(", ":-D", ":-P"}

STOPWORDS = {
    "a","an","the","in","on","at","to","for","of","with","by","from",
    "and","but","or","so",
    "is","are","was","were","be","been","being","do","does","did",
    "have","has","had","will","would","could","should",
    "i","you","he","she","it","we","they",
    "me","him","her","us","them",
    "my","your","his","its","our","their",
    "this","that","these","those","just","also","very",
}

In [ ]:
# TODO 10 — tokenise_14
def tokenise_14(text, keep_emoticons=True):
    tokens = []
    buffer = ""
    i = 0
    while i < len(text):
        if keep_emoticons:
            two_chars = text[i:i+2]
            if two_chars in EMOTICONS_14:
                if buffer.strip(): tokens.append(buffer.strip()); buffer = ""
                tokens.append(two_chars); i += 2; continue
        char = text[i]
        if char == " ":
            if buffer.strip(): tokens.append(buffer.strip()); buffer = ""
        elif char.isalpha() or char.isdigit():
            buffer += char
        i += 1
    if buffer.strip(): tokens.append(buffer.strip())
    return tokens

# TODO 11 — remove_stopwords
def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

# TODO 12 — remove_short_tokens
def remove_short_tokens(tokens, min_length=2):
    return [t for t in tokens if len(t) >= min_length or t in EMOTICONS_14]

# TODO 13 — pipeline
def pipeline(text, keep_emoticons=True, apply_stopwords=True, min_length=2):
    text = text.lower()
    tokens = tokenise_14(text, keep_emoticons=keep_emoticons)
    if apply_stopwords:
        tokens = remove_stopwords(tokens)
    tokens = remove_short_tokens(tokens, min_length=min_length)
    return tokens

In [ ]:
# Part D — Test
for sentence in [
    "The battery life on this phone is absolutely amazing! :)",
    "I can't believe how terrible this product is. Total waste of money.",
    "Not bad, not great... just okay.",
    "Incredible! Best purchase I've made all year :D",
    "Rude staff, cold food. Never coming back.",
]:
    print(f"Input : {sentence}")
    print(f"Output: {pipeline(sentence)}")
    print()

In [ ]:
# Part E — Design decisions
sentence_14 = "This product is NOT as good as I expected. Very disappointing :("
print(f"Input: {sentence_14}\n")
print(f"V1 (stopwords=ON,  emoticons=ON ): {pipeline(sentence_14, True,  True)}")
print(f"V2 (stopwords=OFF, emoticons=ON ): {pipeline(sentence_14, True,  False)}")
print(f"V3 (stopwords=ON,  emoticons=OFF): {pipeline(sentence_14, False, True)}")
print(f"V4 (stopwords=OFF, emoticons=OFF): {pipeline(sentence_14, False, False)}")

**Answers:**

1. V1 is most compact. V2 retains more context including `'expected'`. Both V1/V2 keep `':('` — strong negative signal.
2. `'not'` surviving is **good** — it negates `'good'` and preserves the correct sentiment.
3. `'disappointing'` survives because it is not a stopword and `len >= 2`.
4. `':('` directly encodes negative emotion — losing it in V3/V4 weakens the output significantly.